# Preparing

Устанавливаем и \ или запускаем библиотеки с функциями и всякие полезные фигни. Биоблиотеки нужно установить один раз. Запускать их нужно каждый раз перед тем, как запускать код.

In [ ]:
from scipy.signal import savgol_filter
import mne #eeg analysis
import numpy as np # dataframes and staff
import matplotlib.pyplot as plt # plots
import matplotlib # plots
import PyQt5 # how plots open in you PC
from pathlib import Path # path to files
from mne.channels import find_ch_adjacency # connection btw electrodes
from functools import partial # statistics
import scipy.stats #statistics
import pickle # way to save data
import seaborn as sns # data and stats
from scipy import stats # stats
import statistics # stats
import mne_qt_browser
from mne.datasets import somato
import pathlib

#the following magic line will make all of your figures to be of the chosen size (in inches)
%matplotlib qt5

mne.viz.set_browser_backend('qt')

Чтоб открыть файл, нужно указать к нему путь. Путь:

In [ ]:
path_to_project = Path('/Users/alexmedvedev/Desktop/Vishka/exp/')

In [ ]:
path_to_80_contr = Path('/Users/alexmedvedev/Desktop/Vishka/80_contr')
path_to_80_exp = Path('/Users/alexmedvedev/Desktop/Vishka/80_exp')
path_to_80_saves = Path('/Users/alexmedvedev/Desktop/Vishka/80_saves')
path_to_80_all = Path('/Users/alexmedvedev/Desktop/Vishka/80_all_epoches')

In [ ]:
path_to_saves = Path("/Users/alexmedvedev/Desktop/Vishka/saves_useless")
path_to_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/epoches_useless")
path_to_exp_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/exp_epochs")
path_to_contr_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/contr_epochs")

In [ ]:

path_to_five_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/5-epochs_-1")
path_to_c_five_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/5-epochs-contrl_-1")
path_to_seven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/7-epochs_-1")
path_to_c_seven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/7-epochs-contrl_-1")
path_to_nine_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/9-epochs_-1")
path_to_c_nine_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/9-epochs-contrl_-1")
path_to_eleven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/11-epochs_-1")
path_to_c_eleven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/11-epochs-contrl_-1")

вроде поломанные айдишки у всех файлов поломаны данным образом
'New Segment/': 1,
 'Noise': 2,
 'Stimulus/S  5': 3,
 'Stimulus/S  7': 4,
 'Stimulus/S  9': 5,
 'Stimulus/S 11': 6,
 'Stimulus/S 25': 7,
 'Stimulus/S 27': 8,
 'Stimulus/S 29': 9,
 'Stimulus/S 35': 10,
 'Stimulus/S 51': 11,
 'Stimulus/S 63': 12,
 'Stimulus/S 71': 13,
 'Stimulus/S 91': 14,
 'Stimulus/S111': 15,
 'Stimulus/S211': 16

 НО ОНИ НЕ НУЖНЫ! Надо использовать в виде ID: '5-6', 'contr_5-6' и т.д.

# Load the file

Потом нужно открыть файл(-ы) в этом пути.

In [ ]:
sbj_id = 'Anana' # participant's id. It's easier to load the file having this variable

In [ ]:
raw = mne.io.read_raw_brainvision(path_to_project / f'{sbj_id}.vhdr', eog=['HEOG', 'VEOG'], misc = ['BIP1'], preload = True)

Если в ЭЭГ есть глазные электроды, указывай это в параметре eog. misc не меняй, хз, что это, но надо. Preload позволяет обращаться к файлу со сложными задачами, без него только простые операции.

In [ ]:
raw.info

# Montage

Код не знает, как на голове расположены электроды. Их названия ему ничего не говорят. Надо до него это донести с мопощью подключения монтажа. Грубо говоря -- это карта расположения электродов в заивисимости от названия.

In [ ]:
montage = mne.channels.make_standard_montage("standard_1020")

In [ ]:
#raw.rename_channels({'FP2':'Fp2'})

In [ ]:
raw.set_montage(montage)

In [ ]:
raw.plot_sensors(show_names = True) #What if we would like to see the names

# Filtering

In [ ]:
raw.notch_filter(50)
raw.filter(1,45)

In [ ]:
raw.plot(duration = 100)

# ICA

Individual Component Analysis "объясняет" данные записи, деля их на компоненты. Например, фронтальный компонент (глаза), затылочный, и тд.
Это нужно для извлечения морганий.

In [ ]:
ica = mne.preprocessing.ICA(n_components = 0.99)

In [ ]:
ica.fit(raw, reject_by_annotation=True)

In [ ]:
ica.plot_sources(raw)

In [ ]:
ica.plot_components()

In [ ]:
help(ica.find_bads_eog)


In [ ]:

ica.find_bads_eog(inst = raw, ch_name = ['HEOG', 'VEOG'], measure = 'correlation', threshold = 0.5)

In [ ]:
#find which ICs match the EOG pattern
ica.exclude = []
eog_indices, eog_scores = ica.find_bads_eog(raw, threshold = 0.75)
#ica.exclude = eog_indices
ica.plot_components(eog_indices)

In [ ]:
ica.exclude = [1, 3]
ica.apply(raw)

In [ ]:
raw.plot(duration = 100)


In [ ]:
raw.save(path_to_saves/ f'{sbj_id}_eeg.fif', overwrite=True) #saving preprocessed file

In [ ]:
# ТОДБКО ДЛЯ ЭПОХ 1-80 HZ
raw.save(path_to_80_saves/ f'{sbj_id}_eeg.fif', overwrite=True) #saving preprocessed file

# Epoch

Эпохи -- это вычленение из записи только секунды после метки.

In [ ]:
events, ids = mne.events_from_annotations(raw)

In [ ]:
ids

## Epoch

In [ ]:
dis_ids = {'5-6': 5 or 51, '7-8': 7 or 71, '9-10': 9 or 91, '11-12': 11 or 111, 'contr_5-6': 25, 'contr_7-8': 27, 'contr_9-10': 29 or 35, 'contr_11-12': 211 or 63}
exp_ids = {'5-6': 5 or 51, '7-8': 7 or 71, '9-10': 9 or 91, '11-12': 11 or 111}
contr_ids = {'contr_5-6': 25, 'contr_7-8': 27, 'contr_9-10': 29 or 35, 'contr_11-12': 211 or 63}
five_ids = {'5-6': 5 or 51}
c_five_ids = {'contr_5-6': 25}
seven_ids = {'7-8': 7 or 71}
c_seven_ids = {'contr_7-8': 27}
nine_ids = {'9-10': 9 or 91}
c_nine_ids = {'contr_9-10': 29 or 35}
eleven_ids = {'11-12': 11 or 111}
c_eleven_ids = {'contr_11-12': 211 or 63}

In [ ]:
dis_events = mne.pick_events(events, include = [5,51,7,71,9,91,11,111,25,251,27,271,29,35,211,63])
exp_events = mne.pick_events(events, include = [5,51,7,71,9,91,11,111])
contr_events = mne.pick_events(events, include = [25,27,29,35,211,63])
five_events = mne.pick_events(events, include = [5,51])
c_five_events = mne.pick_events(events, include = [25,251])
seven_events = mne.pick_events(events, include = [7,71])
c_seven_events = mne.pick_events(events, include = [27,271])
nine_events = mne.pick_events(events, include = [9,91])
c_nine_events = mne.pick_events(events, include = [29,35])
eleven_events = mne.pick_events(events, include = [11,111])
c_eleven_events = mne.pick_events(events, include = [211,63])

tmin -- престимульный интервал, часто 0.5 - 0.1. tmax -- постстимульный интервал -- чаще 1. baseline первое число совпадает с престимулом, второе число 0.

In [ ]:
mne.viz.plot_events(dis_events, event_id = dis_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, dis_events, dis_ids, tmin = -0.5, tmax = 1, baseline = (-0.5, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_epoches/ f'{sbj_id}_epo.fif', overwrite=True)

In [ ]:
mne.viz.plot_events(exp_events, event_id = exp_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, exp_events, exp_ids, tmin = -0.5, tmax = 1, baseline = (-0.5, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_exp_epoches/ f'{sbj_id}_epo.fif', overwrite=True)

In [ ]:
mne.viz.plot_events(contr_events, event_id = contr_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, contr_events, contr_ids, tmin = -0.5, tmax = 1, baseline = (-0.5, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_contr_epoches/ f'{sbj_id}_epo.fif', overwrite=True)

ДАЛЬШЕ ТОЛЬКО ДЛЯ ЭПОХ 80 HZ

In [ ]:
mne.viz.plot_events(dis_events, event_id = dis_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, dis_events, dis_ids, tmin = -0.5, tmax = 2, baseline = None, reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_80_all/ f'{sbj_id}_epo.fif', overwrite=True)

In [ ]:
mne.viz.plot_events(exp_events, event_id = exp_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, exp_events, exp_ids, tmin = -0.5, tmax = 2, baseline = None, reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_80_exp/ f'{sbj_id}_epo.fif', overwrite=True)

In [ ]:
mne.viz.plot_events(contr_events, event_id = contr_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, contr_events, contr_ids, tmin = -0.5, tmax = 2, baseline = None, reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_80_contr/ f'{sbj_id}_epo.fif', overwrite=True)

# Evokeds

ERP из эпох.

In [ ]:
evoked_dis = epochs_dis.average()
evoked_dis

In [ ]:
evoked_dis.plot()

In [ ]:
my_evokeds = {}
for condition in epochs_dis.event_id:
    my_evokeds[condition] =  epochs_dis[condition].average()
    
my_evokeds

In [ ]:
my_evokeds['11-12'].plot_topo() # условие в квадратной скобке можно менять

In [ ]:
mne.viz.plot_compare_evokeds(my_evokeds)

In [ ]:
mne.viz.plot_compare_evokeds(
    my_evokeds,
    axes="topo",
    vlines = [0.0, 0.1, 0.3]
) #useless

# ERP

In [ ]:
import pickle

In [ ]:
all_dis_2 = [file for file in path_to_exp_epoches.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
all_dis_epochs_2 = {file.name: None for file in all_dis_2}

for file in all_dis_2: 
    dis_epochs_2 = mne.read_epochs(file)
    all_dis_epochs_2[file.name] = dis_epochs_2

In [ ]:
all_dis_erps_2 = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_dis_epochs_2 = all_dis_epochs_2[name][event_id]
        current_dis_epochs_2.savgol_filter(10.)
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        all_dis_erps_2[name][event_id] = dis_erp_2

In [ ]:
all_dis_erps_2

In [ ]:
grand_averages_2 = {event_id : mne.grand_average([evokeds[event_id] for evokeds in all_dis_erps_2.values()], drop_bads = False) for event_id in [[key for key in val.keys()] for ival, val in enumerate(all_dis_erps_2.values()) if ival == 0][0]}

## PLots

In [ ]:
mne.viz.plot_compare_evokeds({event_id : ga.copy().pick_channels(['C1', 'Cz', 'C2', 'FC1', 'FCz','FC2', 'F1', 'Fz', 'F2']) for event_id, ga in grand_averages_2.items()}, combine = 'mean')

In [ ]:
#mne.viz.plot_compare_evokeds({event_id : ga.copy() for event_id, ga in grand_averages_cntrl_2.items()}, combine = 'mean') - То же. Что за cntrl?

In [ ]:
mne.viz.plot_compare_evokeds(
    {event_id : ga.copy() for event_id, ga in grand_averages_2.items()},
    colors={'5-6' : 0, '7-8' : 1, '9-10':2, '11-12':3},
    linestyles={'5-6' : "solid", '7-8' : "solid", '9-10':"solid", '11-12':"solid"},
    axes="topo",
    vlines = [0.0, 0.1, 0.3, 0.4]
)

In [ ]:
grand_averages_2["11-12"].plot_joint()

In [ ]:
times = np.arange(0.05, 0.65, 0.05)

In [ ]:
grand_averages_2['11-12'].plot_topomap(times = times)